# Worshop on Genome Scale Metabolic Models - CODE-M ###

In this workshop we will use some of what we learned during the lectures to load and explore a genome-scale metabolic model.

There are several tools available for the analysis of GSMMs, but COBRA (Costraints Based Reconstruction and Analysis) is the most used.
https://opencobra.github.io/

COBRA is available for Matlab (COBRA toolbox), in R (cobraR) and for Python (COBRApy). Here we will use the Python version.


## COBRApy

COBRApy is a package written in Python for the analysis of genome-scale metabolic models (GSMMs).
This tool allows to load, manipulate and perform Flux Balance Analysis on a GSMM.

## Installation
Its good practice to first create a conda enviroment and then install COBRApy and other associated packages into that environment.

## Requirements

- Python 3.6 or higher
- pip must be installed
- Jupyter notebooks must be installed


## How to install COBRApy?

See the installation files on canvas for installing and running COBRApy

 
## Quick note about solver
GLPK is the default solver available in python, and this will be enough for this workshop. For more advaced analysis using more better solvers is strongly recommended, such as GUROBI or CPLEX. These additional solvers are free for academic use, but behind a pay wall otherwise.


## Working with a simple GSMM

In this workshop will use the E. coli_core model, which is the model _Escherichia coli_ containing only the central carbon metabolism.
The .xml file for this model is available on Canvas and needs to be downloaded and moved to the same folder where this notebook runs.
For more information about this model you can follow these links:

- https://journals.asm.org/doi/10.1128/ecosalplus.10.2.1
- http://bigg.ucsd.edu/models/e_coli_core


## Lets explore COBRApy

First we need to import cobra. If COBRApy is not installed, donwload the install files (for Mac or Windows) and follow the instructions.

In [ ]:
#!pip install matplotlib

In [ ]:
import cobra #we need cobra to perform FBA and associated analyses
#from cobra.flux_analysis import flux_variability_analysis
#from cobra.flux_analysis import single_gene_deletion
import os
import numpy as np #we need numpy for working with flux values
import matplotlib.pyplot as plt #we will be plotting some charts

Next we need to load the E. coli core metabolic model using the `cobra.io.read_sbml_model()` function. This model contains the core metabolic reactions of central carbon metabolism in e.coli. As such, its a relatively small model, and faster to perform.

In [ ]:
print(os.getcwd())

In [ ]:
print(os.listdir())

In [ ]:
# Loading a model by setting the path to the model location.
model= cobra.io.read_sbml_model('iJO1366.xml')

By reading in the metabolic model using the cobra.io.read_sbml_model() function, we have created a 'Model' object container that contains DictLists objects of genes, metabolites and reactions. This'Model' object represents the entire metbaolic network and allows the user to perfrom operations on the model. Becuase we have given the metbaolic model the object handle 'model', whenever you want to perfrom an operation on the model, you have to provide its object handle.

The reactions, metabolites, and genes attributes of the cobrapy model are a special type of list called a cobra.DictList, and each one is made up of cobra.Reaction, cobra.Metabolite and cobra.Gene objects respectively. We can explore all reactions, metabolites, gene-protein reaction rules etc.


In [ ]:
len(model.genes)

Within a genome scale metabolic model, one or more gene encode for a protein that is involved in a metabolic reacion. Metabolic reactions modify metabolites, changing a substrate to a product. Genes, reactions and metabolties are stored as a DiscList. Just like a regular list, objects in the DictList can be retrieved by index. For example, to get the 30th reaction in the model (at index 29 because of 0-indexing):

We can then access any of these attributes of a metabolic reaction.

Each reaction comes with a set of constraints, also called rection bounds that limit that reactions capacity to carry flux. In the example below, this reaction has a lower bound of -1000, and an upper bound of 1000.

In [ ]:
model.reactions[29].bounds

In [ ]:
#model.reactions[29].lower_bound
model.reactions[29].upper_bound

In [ ]:
for r in model.reactions:
    print(r)

This set of reaction bounds constrain the model, by defining the allowable flux through that reaction.

In [ ]:
for exchange in model.exchanges:
    print(exchange.name, exchange.id, 'bounds: ', exchange.bounds)

In [ ]:
model.medium

What can we tell about this constraints? What is the composition of the medium we are considering?

We can also check what is the current objective function in the model, and if we want we can change it.

In [ ]:
#setting the objective function 
model.objective = "BIOMASS_Ec_iJO1366_WT_53p95M"

In [ ]:
print(model.objective)

# 1. Flux Balance Analysis (FBA) to predict aerobic growth rates of e.coli
This section will demonstrate how to perform Flux Balance Analysis to first predict growth rates of e.coli grown aerobically (in the presence of oxygen). The main carbon source for this simulation is glucose - a 6 carbon molecule. We will need to first create a copy of the model for each simualtion, and then provide a descriptive object handle for each simulation. This ensures we will accessess the correct growth rates and assocaited solution fluxes for each simulation.

create an independent copy of the COBRA model. You can modify the new model without changing the original.

In [ ]:
#create a copy of model and rename model_aerobic. 
model_aerobic = model.copy()
model_aerobic

In [ ]:
#### copying the model also copies the objective reaction to optimise.
print(model_aerobic.objective)

We have now two version of the same model that will allow us to simulate one model under aeroboc conditions, without changing the conditions of the original model.

The next step is to constrain the maximum glucose and oxygen uptake rate to a realisitic values of 8.5 mmol gDW-1 hr-1 (millimoles per gram dry cell weight per hour, the default flux units used in the COBRApy Toolbox). This changes the lower bound and upper bound of the glucose exchange reaction to -8.5, a biologically realistic uptake rate.  By convention, exchange reactions are written as export reactions (e.g. ‘EX_glc(e) <==>’), so import of a metabolite is a negative flux.

## but why set oxygen uptake to 18.5 mmol gDW-1 hr-1 and glucose to 8

In [ ]:
#change bounds of glucose uptake to force glucose uptake
model_aerobic.reactions.EX_glc__D_e.bounds = (-8, -0)

To allow unlimited oxygen uptake to simulate aerobic growth, we can unconstrain the bounds of the oxygen uptake reaction to such a large number so that it is practically unbounded.

In [ ]:
model_aerobic.reactions.EX_o2_e.bounds = (-18.5, 0)

we can check these constraints have been incorportated into the model by using the .medium command

In [ ]:
#check glucose and oxygen have been cosntrained respectively.
model_aerobic.medium

double check what the objective reaction to be optimised is

In [ ]:
print(model_aerobic.objective)

Now perform FBA with maximization of the biomass reaction as the objective using the .optimize() function. We will first give the output of this solution a mathcing object handle.

In [ ]:
solution_aerobic = model_aerobic.optimize() # this command runs FBA on model_aerobic, and saves the solution in solution_aerobic

Simulations using flux balance analysis (FBA) can be performed with the command model.optimize(). By default, this function maximizes flux through the model’s objective reaction, though it can also be configured to minimize it. The function returns a Solution object that contains the results of the optimization. Important attributes of the Solution object include:

-  objective_value: the value of the objective function (e.g., predicted growth rate) at the solution
-  status: the status returned by the linear programming solver ('optimal', 'infeasible', etc.)

For example, after calling model.optimize(), the status attribute should be 'optimal' if the solver returned a valid solution, and objective_value gives the predicted flux through the objective reaction.

In [ ]:
print('Status: ',solution_aerobic.status)
print('Aerobic growth rate: %.4f' % solution_aerobic.objective_value)

The .summary() method prints a high-level overview of the FBA solution after optimization.
It typically includes:
-  Objective value; The optimized value of the objective function. Often the biomass growth rate
-  Uptake fluxes (inputs); Metabolites being consumed,Their flux values and Percent contribution
-  Secretion fluxes (outputs); Metabolites being produced and secreted


## Is this predicted growth rate realisitc?
Fischer et al (2004) report growth rates that range from 0.61 - 0.82. So our predicetd growth rate falls between these values

In [ ]:
model_aerobic.summary()

The result returned by solution.objective_value provides the value of the objective function (Z) as 0.7866.  This means that the model predicts an optimised aerobic growth rate of 0.7866 hr-1,  To achive ths, glucose, ammonium, oxygen and inorganic phosphate were all imported, leading to the secretion of CO2, H20 and hydrogen

To access the fluxes associated to the reactions within the solution object use solution.fluxes

In [ ]:
solution_aerobic.fluxes

Or to print the first 10 reactions and associated fluxes nicely...

In [ ]:
for rxn in model.reactions[:10]:
    print(f"{rxn.id:20s}: {solution_aerobic.fluxes[rxn.id]: .3f}") #20s refers to the seperator space between rxn.id and :/

It is much more descritive, and visual to diplsay this DictList of reaction fluxes as a metabolic network. We can do this for e.coli using Escher

## Draw metabolic maps with Escher
Escher is a web application for visualising data on biological pathway maps. install Escher in a notebook using !pip install cobra escher

More information:

King, Z. A., Dräger, A., Ebrahim, A., Sonnenschein, N., Lewis, N. E., & Palsson, B. O. (2015). Escher: A Web Application for Building, Sharing, and Embedding Data-Rich Visualizations of Biological Pathways. PLOS Computational Biology, 11(8), e1004321. http://doi.org/10.1371/journal.pcbi.1004321
Documentation https://escher.readthedocs.org/en/latest/getting_started.html

In [ ]:
!pip install cobra escher #only run this line once to install escher, then after comment out
import escher
from escher import Builder #Builder is used to create metabolic flux maps

List available metabolic maps in the Escher package:

In [ ]:
escher.list_available_maps()

list available metabolic model orgnaisms in Escher

In [ ]:
escher.list_available_models()

To use Escher, first select a built in metabolic map you want to map the fluxes onto.

In [ ]:
#map_name is the identifier for the built-in E. coli core metabolic map in Escher.
map_name = "iJO1366.Central metabolism"

Then use Builder to create an interactive Escher map. To use Builder, you need to specify the following arguments...

-  model=model → the COBRApy model used for FBA.
-  reaction_data=solution.fluxes.to_dict() → provides the calculated fluxes following optimisation for visulisation.
-  hide_no_data=False → reactions with no flux data will still appear on the map (instead of being hidden).

In [ ]:
builder = Builder(
    model=model_aerobic,
    map_name=map_name,
    reaction_data=solution_aerobic.fluxes.to_dict(),
    hide_no_data=False
)

The metabolic map with mapped fluxes will be output as an HTML file, and saved in the current working directory.

In [ ]:
# Save to an HTML file
html_file = "aerobic_e_coli.html" #give a concise yet descriptive file name
builder.save_html(html_file)

# Print full path to display where it is saved too.
full_path = os.path.abspath(html_file)
print(f"Escher map ready: {full_path}")

Inspection of the flux distribution shows that there is high flux in the glycolysis, pentose phosphate, TCA cycle, and oxidative phosphorylation pathways, and that no organic by-products are secreted. 

The following analysis shows how changing glucose constraints predict growth. Increasing glucose availability may increase growth initially, but the benefit can diminish when oxygen or another constraint becomes limiting.

In [ ]:
growth_rates = []
glucose_range = np.sort(
    np.append(np.arange(0, 21, 2), 11))

with model_aerobic:
    model_aerobic.reactions.EX_o2_e.bounds = (-18.5, 0)
    glc = model_aerobic.reactions.EX_glc__D_e

    for uptake in glucose_range:
        glc.bounds = (-float(uptake), 0)
        solution = model_aerobic.optimize()

        growth_rates.append(
            solution.objective_value
            if solution.status == "optimal"
            else np.nan
        )

plt.plot(glucose_range, growth_rates, marker="o")
plt.xlabel("Maximum glucose uptake (mmol/gDW/h)")
plt.ylabel("Predicted growth rate (h⁻¹)")
plt.title("Growth vs glucose availability (maximum O₂ uptake = 18.5)")
plt.grid(True)
plt.show()

In [ ]:
growth_by_glucose = dict(zip(glucose_range, growth_rates))

print(
    f"Growth rate when glucose uptake is 11 h⁻¹: "
    f"{growth_by_glucose[11]:.3f} h⁻¹"
)

With oyxgen uptake fixed at 18.5 mmol gDW-1 hr-1, growth rate increases steadily as glucose uptake increases. At an glucose uptake of about 10 mmol gDW-1 hr-1, growth rate decreases.

## Growth on alternate substrates
Just as FBA was used to calculate growth rates of E. coli on glucose, it can also be used to simulate growth on other substrates. The core E. coli model contains exchange reactions for 13 different organic compounds, each of which can be used as the sole carbon source under aerobic conditions. We will predict the growth of e.coli on succinate as the sole carbon source in the presence of oxygen.



In [ ]:
#unconstrain oxygen to promote aerobic growth, constrain gluose uptake to zero, and constrain succinate to force uptake
model_aerobic_succinate = model_aerobic.copy()
model_aerobic_succinate.reactions.EX_o2_e.bounds = (-1000, 1000) 
model_aerobic_succinate.reactions.EX_glc__D_e.bounds = (-0, -0)
model_aerobic_succinate.reactions.EX_succ_e.bounds = (-8,-8)
model_aerobic_succinate.medium #you can see succinate and oxygen is now part of the growth medium

In [ ]:
solution_aerobic_succinate = model_aerobic_succinate.optimize() # performs FBA on model_aerobic_succinate and output solution to a new object
solution_aerobic_succinate.objective_value

The growth rate on succinate as a sole carbon source is 0.39 hr-1. This is much lower than aerobic growth on glucose which was predicted to be 0.7496 hr-1

In [ ]:
#Can e.coli grow on succinate without oxygen?
model_anaerobic_succinate =  model_aerobic_succinate.copy()
#lets first check growth medium
model_anaerobic_succinate.medium

In [ ]:
#need to remove oxygen from the growth medium
model_anaerobic_succinate.reactions.EX_o2_e.bounds = (-0, 0)
model_anaerobic_succinate.medium #check to see that succinate. glucose and oxygen are removed from medium

In [ ]:
solution_anaerobic_succinate = model_anaerobic_succinate.optimize() # performs FBA on model_anaerobic and output solution to a new object
model_anaerobic_succinate.summary()

The solution is infeasible: FBA predicts that growth is not possible on succinate under anaerobic conditions, there is no feasible solution. This is because the maximum amount of ATP that can be produced in this condition is less than the minimum amount of ATP required for the organism to grow

Now that you have come to the end of this tutorial on using COBRApy, you should be able to load a model, set an objective function, constrain the model, perfrom FBA and FVA and visulsie the results in Escher. You should also be able to perform a robustness analyses, a phenotypic phase plane analysis, reaction and gene knockouts and reaction and gene essential analysis.

In the Rest of this workshop I want you to answer the following quesitons:

# Exercise on Genome Scale Metabolic Models to predict maximum ethanol production rate from S.cerevisiae

In this exercise you should:

1. Load the iMM904.xml model provided. This is an genome scale metabolic model of Saccharomyces cerevisiae.  You can read more about this genome scale metabolic model reconstruction here: https://pubmed.ncbi.nlm.nih.gov/19321003/

2. How many reactions are present in the model?

3. How many genes are present in the model?

4. How many metabolites are present in the model?

5. Print all the metabolite exchange reactions

6. Which nutrients is the model allowed to take up?

7. Using FBA, simulate the maximum aerobic growth rate predicted?

8. Plot the fluxes using Escher

9. Predict the effect of how increasing oxygen availability influcences predicted growth rate. If growth becomes limited, what could be limiting growth and how could you test that explanation? 